# 40 · Phase 4: anonymous multi-camera analytics

Phase 4 joins calibrated cameras on one ground plane. WILDTRACK gives seven synchronised, calibrated views of a busy square, with every person located on a 2.5 cm grid. A person detector fine-tuned on the training frames runs in every view; foot points are projected to the ground and fused by visibility-normalised voting; people are tracked on the ground plane with anonymous, ephemeral ids; and the system reports only aggregates: zone counts, line crossings, dwell times and occupancy. Faces are blurred before anything is stored or drawn.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot` and `mrriandmstique/wildtrack-multi-camera-tracking` |
| **Expected runtime** | about 2 hours on 2 x T4 |
| **Produces** | `results/phase4/`: detection, tracking and count tables, the operator board and a clip |

In [ ]:
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
                if (Path(p).parent / "src" / "cffm").is_dir()]
        hits.sort(key=lambda h: any((h.parent / d).exists() for d in ("runs", "data")))
        zips = [Path(p) for k in range(1, 5) for p in glob.glob("/kaggle/input" + "/*" * k + ".zip")
                if "cffm" in Path(p).name]
        if hits:
            shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        elif zips:
            shutil.unpack_archive(str(zips[0]), "/kaggle/working")
        assert PROJECT.exists(), "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
        for p in [PROJECT, *PROJECT.rglob("*")]:
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle", "pytest", "lap>=0.5.12", "imageio-ffmpeg", "motmetrics"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)
env = cenv.setup()
plan = pipeline.load_plan(env.project)

In [ ]:
from pathlib import Path
import json
import cv2, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from cffm import multicam as mc, hud, showcase as sc
from ultralytics import YOLO

INPUT = Path("/kaggle/input") if ON_KAGGLE else env.project / "data" / "raw"
ROOT = next(p.parent for p in INPUT.rglob("calibrations") if (p.parent / "annotations_positions").exists())
views = mc.load_views(ROOT)
out = env.project / "results" / "phase4"
out.mkdir(parents=True, exist_ok=True)
val = [f for f in mc.FRAMES if mc.TEST_FROM - 200 <= f < mc.TEST_FROM]
test = [f for f in mc.FRAMES if f >= mc.TEST_FROM]
print("WILDTRACK:", ROOT, "| validation frames:", len(val), "| test frames:", len(test))

## 1. Geometry check

Every annotated person projected through the calibration must land on the feet of their box in every view that sees them.

In [ ]:
errs = []
for p in mc.load_annotations(ROOT, 0):
    xy = mc.position_to_cm([p["positionID"]])
    for v in p["views"]:
        if v["xmin"] >= 0:
            uv = views[v["viewNum"]].to_image(xy)[0]
            errs.append(np.linalg.norm(uv - [(v["xmin"] + v["xmax"]) / 2, v["ymax"]]) / (v["ymax"] - v["ymin"]))
print(f"median foot error {np.median(errs) * 100:.1f}% of person height over {len(errs)} boxes")

## 2. Fine-tune the per-view detector

YOLO26-m, COCO-pretrained, fine-tuned on the per-view boxes of the first 320 frames. Frames 1600 to 1795 validate, and the last 40 frames, the standard 10 % test split, are never seen in training or tuning.

In [ ]:
ds = mc.export_yolo(ROOT, env.data / "wildtrack")
print("exported", ds)
run = env.runs / "wildtrack_yolo26m"
best, last = run / "weights" / "best.pt", run / "weights" / "last.pt"
if not best.exists() or last.exists() and not (run / "done").exists():
    m = YOLO(str(last) if last.exists() else "yolo26m.pt")
    m.train(data=str(env.data / "wildtrack" / "data.yaml"), imgsz=1280, epochs=40, batch=8, device=env.device,
            project=str(env.runs), name="wildtrack_yolo26m", exist_ok=True, close_mosaic=5, resume=last.exists(),
            plots=False, workers=4)
    (run / "done").touch()
det_models = {"zero-shot YOLO26-m": YOLO("yolo26m.pt"), "fine-tuned YOLO26-m": YOLO(str(best))}

## 3. Detect in every view

In [ ]:
cache = out / "detections.npz"
if cache.exists():
    D = {k: v.item() for k, v in np.load(cache, allow_pickle=True).items()}
else:
    D = {}
    for name, m in det_models.items():
        D[name] = {}
        for f in val + test:
            ims = [str(mc.image_path(ROOT, c, f)) for c in range(len(views))]
            res = m.predict(ims, imgsz=1280, conf=0.05, classes=[0], verbose=False, half=True)
            D[name][f] = [np.column_stack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(),
                                           r.boxes.cls.cpu().numpy()]) for r in res]
    np.savez(cache, **{k: np.array(v, dtype=object) for k, v in D.items()})
print({k: len(v) for k, v in D.items()})

## 4. Ground-plane detection

Fusion parameters are chosen on the validation frames only, then fixed. MODA, MODP, precision and recall at the standard 0.5 m radius on the test frames.

In [ ]:
def ground(name, f, **kw):
    per = [mc.foot_points(views[c], D[name][f][c]) for c in range(len(views))]
    return mc.vote(views, [p for p, _ in per], [s for _, s in per], **kw)[0]

grid = [dict(radius=r, min_ratio=q, min_score=s) for r in (40, 60, 80) for q in (0.3, 0.5) for s in (0.15, 0.3, 0.45)]
gt = {f: mc.gt_positions(ROOT, f) for f in val + test}
res, chosen = {}, {}
for name in D:
    chosen[name] = max(grid, key=lambda kw: mc.moda([ground(name, f, **kw) for f in val], [gt[f] for f in val])["MODA"])
    res[name] = mc.moda([ground(name, f, **chosen[name]) for f in test], [gt[f] for f in test])
(pd.DataFrame(res).T[["MODA", "MODP", "precision", "recall"]] * 100).round(1)

## 5. Anonymous tracking and counts

Tracks live on the ground plane only, with ids that exist for the life of a track and are never linked to appearance. The exit test of Phase 4 is counts that agree with a manual count; here the annotated positions are that count.

In [ ]:
name = "fine-tuned YOLO26-m"
frames = val + test
trk = mc.GroundTracker(gate=80, max_miss=3, min_hits=2)
track_frames, pred = {}, {}
for f in frames:
    pred[f] = ground(name, f, **chosen[name])
    track_frames[f] = [(t.tid, *t.xy) for t in trk.update(pred[f], f // 5)]
zones = mc.default_zones()
mot = mc.ground_mot({f: track_frames[f] for f in test}, ROOT, test)
counts = mc.count_agreement([pred[f] for f in test], [gt[f] for f in test], zones)
tracks = trk.all_tracks()
ana = {"tracking": mot, "counts": counts, "crossings of y = 0": mc.line_crossings(tracks, 0.0),
       "dwell": mc.dwell_seconds(tracks, zones, fps=2)}
pd.DataFrame({"MOTA": [mot["mota"]], "IDF1": [mot["idf1"]], "ID switches": [mot["num_switches"]],
              "count MAE": [counts["count_mae"]], "mean people": [counts["mean_people"]],
              "relative count error": [counts["count_rel_err"]]}).round(3)

## 6. The operator board

Seven views with faces blurred at ingest, beside the ground plane seen from above: every anonymous track with its trail, the zones and their live counts.

In [ ]:
SCALE = 0.35

def to_px(xy):
    return ((np.asarray(xy, float) - mc.ORIGIN_CM) * SCALE).astype(int)

def bev(f):
    W, H = int(mc.GRID[0] * mc.CELL_CM * SCALE), int(mc.GRID[1] * mc.CELL_CM * SCALE)
    img = np.full((H, W, 3), (20, 16, 14), np.uint8)
    for z in zones:
        (x0, y0), (x1, y1) = to_px([z.x[0], z.y[0]]), to_px([z.x[1], z.y[1]])
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1) - 1, int(y1) - 1), (76, 66, 60), 1)
        cv2.putText(img, f"{z.name.upper()}  {int(z.contains(pred[f]).sum())}", (int(x0) + 8, int(y0) + 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (230, 230, 230), 1, cv2.LINE_AA)
    for t in tracks:
        path = [p for p in t.path if p[0] <= f // 5][-12:]
        if path and path[-1][0] == f // 5:
            pts = to_px([p[1:] for p in path])
            cv2.polylines(img, [pts.reshape(-1, 1, 2)], False, (0, 160, 255), 2, cv2.LINE_AA)
            cv2.circle(img, (int(pts[-1][0]), int(pts[-1][1])), 7, (0, 204, 255), 2, cv2.LINE_AA)
            cv2.putText(img, f"{t.tid:03d}", (int(pts[-1][0]) + 9, int(pts[-1][1]) - 9), cv2.FONT_HERSHEY_SIMPLEX,
                        0.45, (0, 204, 255), 1, cv2.LINE_AA)
    return img

def board(f):
    tiles = []
    for c in range(len(views)):
        im = cv2.imread(str(mc.image_path(ROOT, c, f)))
        d = D[name][f][c]
        d = d[d[:, 4] >= 0.3]
        im = mc.anonymize(im, d[:, :4])
        for x1, y1, x2, y2, *_ in d:
            cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 204, 255), 3)
        tiles.append(cv2.resize(im, (640, 360)))
    left = np.concatenate([np.concatenate(tiles[:4], 1), np.concatenate(tiles[4:] + [np.zeros_like(tiles[0])], 1)], 0)
    b = bev(f)
    right = cv2.resize(b, (int(left.shape[0] * b.shape[1] / b.shape[0]), left.shape[0]))
    return np.concatenate([left, right], 1)[..., ::-1]

boards = [board(f) for f in test]
hud.save_video(boards, out / "multicamera.mp4", fps=2, width=1920)
sc.show(boards[len(boards) // 2], width=1400)

In [ ]:
hm = mc.heatmap(np.concatenate([pred[f] for f in frames if len(pred[f])]))
plt.figure(figsize=(3, 7))
plt.imshow(hm, cmap="inferno")
plt.axis("off")
plt.title("occupancy")
plt.savefig(out / "heatmap.png", dpi=150, bbox_inches="tight")
(out / "phase4_summary.json").write_text(json.dumps({"detection": res, "fusion_params": chosen, **ana},
                                                    indent=2, default=float))
print("saved", sorted(p.name for p in out.iterdir()))

In [ ]:
if env.platform == "kaggle":
    shutil.rmtree(env.data, ignore_errors=True)